# Entrenar "oye matix" con TU voz

Reentrena el wake word usando tus grabaciones reales (subidas desde la app) en vez de voces sinteticas. Pasos: 1) Entorno GPU (T4). 2) Corre las celdas EN ORDEN. En la celda 4 subes el zip `oye_matix_muestras.zip` que te paso Claude. Al final descargas `oye_matix.onnx`.


## 1. Instalar openWakeWord + extras

In [ ]:
# Clona openWakeWord e instala TODO lo necesario (incluye los paquetes que faltaban:
# pronouncing, onnx). Tarda ~3-4 min. Las advertencias de pip (transformers/fsspec/
# huggingface-hub) son RUIDO, no rompen nada.
!git clone https://github.com/dscripka/openWakeWord
%pip install -q -e ./openWakeWord
%pip install -q piper-tts pronouncing onnx onnxscript
%pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
  speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
  acoustics==0.2.6 'datasets[audio]==2.14.6'
print('instalacion lista')

## 2. Arreglos de compatibilidad + modelos base

In [ ]:
# Arreglos de compatibilidad (todos juntos). Necesarios para que train.py corra.
import glob, re, os, sys

# a) torch-audiomentations llama torchaudio.set_audio_backend() AL IMPORTARSE, y
#    torchaudio nuevo la elimino. Editamos el archivo SIN importar el paquete.
for f in glob.glob('/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py'):
    s = open(f).read()
    open(f, 'w').write(re.sub(r'torchaudio\.set_audio_backend\([^)]*\)', 'pass', s))
print('a) torch-audiomentations parcheado')

# b) train.py importa 'generate_samples' (de piper-sample-generator) aunque solo se
#    use con --generate_clips (que NO usamos). Creamos un modulo vacio.
d = '/content/openWakeWord/piper-sample-generator'
os.makedirs(d, exist_ok=True)
open(d + '/generate_samples.py', 'w').write(
    'def generate_samples(*a, **k):\n    raise RuntimeError("no se usa en modo augment")\n')
print('b) dummy generate_samples creado')

# c) Modelos base de openWakeWord (melspectrograma + embedding): con pip no vienen.
sys.path.insert(0, '/content/openWakeWord')   # para poder importar openwakeword aqui
import openwakeword.utils
openwakeword.utils.download_models()
print('c) modelos base:', len(glob.glob('/content/openWakeWord/openwakeword/resources/models/*.onnx')), '.onnx')

## 3. Corpus de negativos (2000 h) + RIR (reverb)

In [ ]:
# Negativos: RIR (reverb) + ~2000 h de features precomputadas + features de validacion.
import os, glob, shutil
from huggingface_hub import hf_hub_download, snapshot_download
os.makedirs('mit_rirs', exist_ok=True)
snapshot_download('davidscripka/MIT_environmental_impulse_responses', repo_type='dataset', local_dir='mit_rirs')
# Aplanar los RIR .wav: vienen en subcarpeta y openWakeWord le pasaria la carpeta a
# torchaudio.load (que falla). Los copiamos planos a /content/rirs_flat.
os.makedirs('/content/rirs_flat', exist_ok=True)
for w in glob.glob('/content/mit_rirs/**/*.wav', recursive=True):
    shutil.copy(w, '/content/rirs_flat/' + os.path.basename(w))
print('RIR wavs:', len(glob.glob('/content/rirs_flat/*.wav')))
hf_hub_download('davidscripka/openwakeword_features', 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
               repo_type='dataset', local_dir='/content')
hf_hub_download('davidscripka/openwakeword_features', 'validation_set_features.npy',
               repo_type='dataset', local_dir='/content')
print('negativos listos')

## 4. Subir TUS grabaciones y repartir train/test

In [ ]:
# === SUBE AQUI el zip de TUS grabaciones (oye_matix_muestras.zip) ===
# Trae carpetas positivo/ y negativo/. Las repartimos en train/test.
import os, glob, zipfile, shutil, random
from google.colab import files
ROOT = '/content/my_custom_model/oye_matix'
for d in ['positive_train','positive_test','negative_train','negative_test']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
    for f in glob.glob(f'{ROOT}/{d}/*.wav'): os.remove(f)
print('Elige el archivo oye_matix_muestras.zip ...')
up = files.upload()
zname = [n for n in up if n.lower().endswith('.zip')][0]
shutil.rmtree('/content/mi_voz', ignore_errors=True)
with zipfile.ZipFile(zname) as z: z.extractall('/content/mi_voz')
pos = sorted(glob.glob('/content/mi_voz/**/positivo/*.wav', recursive=True))
neg = sorted(glob.glob('/content/mi_voz/**/negativo/*.wav', recursive=True))
print('positivos:', len(pos), '| negativos:', len(neg))
assert pos, 'No encontre positivos en el zip'
random.seed(7); random.shuffle(pos); random.shuffle(neg)
def split(lst, frac=0.2):
    if len(lst) < 2: return lst, lst[:1]
    k = max(1, int(len(lst)*frac)); return lst[k:], lst[:k]
ptr, pte = split(pos); ntr, nte = split(neg)
def put(src, dst):
    for s in src: shutil.copy(s, f'{ROOT}/{dst}/' + os.path.basename(s))
put(ptr,'positive_train'); put(pte,'positive_test')
put(ntr,'negative_train'); put(nte,'negative_test')
print('positive_train', len(ptr), '| positive_test', len(pte))
print('negative_train', len(ntr), '| negative_test', len(nte))
for f in glob.glob(ROOT+'/*_features*.npy'): os.remove(f)
print('LISTO. Ahora corre las celdas de config, aumentar y entrenar.')


## 5. Config YAML (aumentacion x4 para tus clips reales)

In [ ]:
cfg = '''
target_phrase: ["oye matix"]
model_name: "oye_matix"
output_dir: "/content/my_custom_model"
piper_sample_generator_path: "./piper-sample-generator"
n_samples: 320
n_samples_val: 64
tts_batch_size: 50
rir_paths: ["/content/rirs_flat"]
background_paths: []
background_paths_duplication_rate: []
custom_negative_phrases: []
augmentation_rounds: 4
augmentation_batch_size: 4
batch_n_per_class:
  positive: 50
  adversarial_negative: 50
  ACAV100M_sample: 256
feature_data_files:
  ACAV100M_sample: "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
false_positive_validation_data_path: "/content/validation_set_features.npy"
model_type: "dnn"
layer_size: 32
steps: 20000
max_negative_weight: 1500
target_accuracy: 0.6
target_recall: 0.25
target_false_positives_per_hour: 0.2
'''
open('/content/oye_matix.yaml', 'w').write(cfg)
print('config escrita')

## 6. Aumentar + calcular features

In [ ]:
# subprocess CAPTURANDO la salida (sin capturar, el error de train.py no se ve en
# la celda). Si returncode != 0, la celda se pone roja y ademas se ve el error.
import subprocess
print('Aumentando (unos minutos)...')
r = subprocess.run('cd openWakeWord && python openwakeword/train.py --training_config /content/oye_matix.yaml --augment_clips', shell=True, capture_output=True, text=True)
print(r.stdout[-3000:])
print('===== ERROR (STDERR) =====')
print(r.stderr[-3000:])
assert r.returncode == 0, 'El AUGMENT fallo: mira el STDERR de arriba.'
print('AUGMENT OK')

## 7. Entrenar el clasificador (~3 min)

In [ ]:
import subprocess, glob
print('Entrenando...')
r = subprocess.run('cd openWakeWord && python openwakeword/train.py --training_config /content/oye_matix.yaml --train_model', shell=True, capture_output=True, text=True)
print(r.stdout[-3000:])
print('===== ERROR (STDERR) =====')
print(r.stderr[-3000:])
assert r.returncode == 0, 'El TRAIN fallo: mira el STDERR de arriba.'
ms = glob.glob('/content/my_custom_model/**/oye_matix.onnx', recursive=True)
assert ms, 'Entreno pero no encuentro oye_matix.onnx'
print('TRAIN OK ->', ms[0])

## 8. Verificar interfaz ONNX [1,16,96] -> [1,1]

In [ ]:
import onnx, glob
p = glob.glob('/content/my_custom_model/**/oye_matix.onnx', recursive=True)[0]
m = onnx.load(p)
print('archivo:', p)
print('inputs :', [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print('outputs:', [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])
print('Debe ser entrada [1, 16, 96] y salida [1, 1].')

## 9. Consolidar (1 archivo) + descargar

In [ ]:
# Consolida en UN solo archivo (pesos embebidos) y descarga + Drive.
import glob, shutil, os, onnx
src = glob.glob('/content/my_custom_model/**/oye_matix.onnx', recursive=True)[0]
m = onnx.load(src)  # carga con sus pesos (misma carpeta)
onnx.save_model(m, '/content/oye_matix.onnx', save_as_external_data=False)
print('consolidado:', round(os.path.getsize('/content/oye_matix.onnx')/1024,1), 'KB (un solo archivo)')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy('/content/oye_matix.onnx', '/content/drive/MyDrive/oye_matix.onnx')
    print('GUARDADO en tu Google Drive (Mi unidad) como oye_matix.onnx')
except Exception as e:
    print('No pude montar Drive:', e)
from google.colab import files
files.download('/content/oye_matix.onnx')
